In [1]:
import json
with open("intermediate_data/form_data_with_prompts.json", "r", encoding="utf-8") as f:
    form_data_with_prompts_dict = json.load(f)

In [6]:
def _render_prompt_content(node: Any, indent: int) -> str:
    pad = "  " * indent
    output: list[str] = []

    # Case: leaf node (value + prompt)
    if (
        isinstance(node, dict)
        and "value" in node
        and "prompt" in node
        and len(node) == 2
    ):
        output.append(f"{pad}value: {node['value']}")
        if node["prompt"]:
            output.append(f"{pad}instruction: {node['prompt']}")
        return "\n".join(output)

    # Case: nested dict
    if isinstance(node, dict):
        for key, value in node.items():
            if key == "prompt":
                continue
            output.append(f"{pad}{key}:")
            output.append(_render_prompt_content(value, indent + 1))
        return "\n".join(output)

    # Fallback (should not normally happen)
    return f"{pad}value: {node}"

In [7]:
def _build_single_section_prompt(
    section_name: str,
    section_body: Any,
) -> str:
    lines: list[str] = []

    # ---- Header ----
    lines.append(
        f"You are completing the European Court of Human Rights application form.\n"
        f"Section: {section_name}\n"
        f"{'-' * 40}"
    )

    # ---- Optional section-level instruction ----
    if isinstance(section_body, dict) and "prompt" in section_body:
        lines.append(
            f"\nSection instruction:\n{section_body['prompt']}"
        )

    # ---- Instructions ----
    lines.append(
        "\nUse the extracted values below.\n"
        "If values are incomplete, noisy, or incorrect, correct them.\n"
        "If information is missing and cannot be inferred, use null.\n"
    )

    # ---- Content ----
    lines.append(_render_prompt_content(section_body, indent=0))

    # ---- Output constraint ----
    lines.append(
        "\nReturn ONLY valid JSON for this section.\n"
        "Do not include explanations or comments."
    )

    return "\n".join(lines)

In [9]:
from typing import Dict, Any


def build_section_prompts(
    prompted_output: Dict[str, Any]
) -> Dict[str, str]:
    """
    Build one LLM-ready prompt per TOP-LEVEL ECHR section.

    Returns:
        {
            "A. Applicant": "<prompt text>",
            ...
        }
    """

    section_prompts: Dict[str, str] = {}

    for section_name, section_body in prompted_output.items():
        prompt = _build_single_section_prompt(
            section_name=section_name,
            section_body=section_body,
        )
        section_prompts[section_name] = prompt

    return section_prompts


In [10]:
build_section_prompts(form_data_with_prompts_dict)

{'A. Applicant': 'You are completing the European Court of Human Rights application form.\nSection: A. Applicant\n----------------------------------------\n\nUse the extracted values below.\nIf values are incomplete, noisy, or incorrect, correct them.\nIf information is missing and cannot be inferred, use null.\n\nA.1 Individual:\n  Surname:\n    value: D\n    instruction: Enter the applicant’s family name as shown on official documents.\n  First name(s):\n    value: A\n    instruction: Enter all given names of the applicant.\n  Date of birth:\n    value: 1)/2/1/)2/)1/9) 8/9 e.g. 31/12/1960 D D M M Y Y Y Y\n    instruction: Provide the applicant’s date of birth (DD/MM/YYYY).\n  Place of birth:\n    value: Thilisi, Georgia\n    instruction: State the city and country where the applicant was born.\n  Nationality:\n    value: Georgia\n    instruction: State the applicant’s current nationality.\n  Address:\n    value: . St. Ni..., building ..., apartment N.., 0000, Thilisi, Georgia\n    in